# CSE754 - PHASE 9
## Multi-Seed Validation + Confidence Diagnostic
Author: Sadia Ruhama — MSc Proposal (CSE 754)

**Why this notebook exists.** Phase 7's round-0 acquisition comparison showed
`class_balanced_bald_OLD` and `class_balanced_bald_NEW` landing on the **exact same value**
(30.0% rare-capture). That's not coincidence: `class_balanced_bald_v2` only differs from the
original when the model's predicted probability distribution is *soft* (spread across classes).
If predictions are already near one-hot (overconfident), `(mean_probs * inv_freq).sum()`
mathematically collapses to `inv_freq[argmax]` — the two formulas become identical. Since Phase 4
already found this model tends to overfit and become overconfident quickly, the two acquisition
functions may simply never get a chance to diverge.

**Section 3** below adds the missing diagnostic: printing the mean max-softmax-probability over
the unlabeled pool at round 0, *before* acquisition. If that number is high (>0.85-0.9), it
confirms the mechanism above directly, rather than leaving it as a hypothesis.

**Sections 4 onward** re-run Phase 7's same-batch experiment across **3 seeds** (42, 123, 2024).
A single seed cannot distinguish "class-balanced BALD genuinely beats baselines" from "this
particular random seed's labeled/unlabeled split happened to favor it" — especially with only
12-15 rare-class examples total. This notebook reports **mean +/- std** across seeds for every
headline metric, which is the minimum bar for either a confident course-report claim or, later, a
publication claim.

Cross-batch multi-seed re-runs are provided as an **optional, not-run-by-default** cell at the end
(Section 7) since it roughly doubles total compute — same-batch is prioritized because it's where
the acquisition-function claim lives.

Recommended runtime: Colab GPU (T4). Expect ~3x Phase 7's same-batch runtime (three seeds).

## 1. Setup — installs and Google Drive

In [ ]:
!pip install -q -U transformers peft accelerate torchao scikit-learn matplotlib timm

import os, json, random, time
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision.datasets import ImageFolder

import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, average_precision_score, balanced_accuracy_score

from transformers import AutoImageProcessor, AutoModel
from peft import LoraConfig, get_peft_model

from google.colab import drive
drive.mount('/content/drive')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

PROJECT_ROOT = '/content/drive/MyDrive/cse754'
OUT_DIR = os.path.join(PROJECT_ROOT, 'outputs')
CKPT_DIR = os.path.join(OUT_DIR, 'checkpoints')
DATA_DIR_SAMEBATCH = os.path.join(PROJECT_ROOT, 'data_v2')
DATA_DIR_CROSSBATCH = os.path.join(PROJECT_ROOT, 'data_crossbatch_v2')
for d in [OUT_DIR, CKPT_DIR]:
    os.makedirs(d, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cuda


## 2. Config — SEEDS list is the only structural addition vs. Phase 7

In [ ]:
@dataclass
class Config:
    image_size: int = 224
    num_classes: int = 8
    rare_class_idx: int = 2   # BV2, per Phase 6's printed class-index mapping

    backbone_name: str = 'facebook/dinov2-small'
    lora_r: int = 8
    lora_alpha: int = 16
    lora_dropout: float = 0.05
    lora_target_modules: Tuple[str, ...] = ('query', 'value')

    hidden_dim: int = 256
    mc_dropout_p: float = 0.3
    mc_passes: int = 20

    n_al_rounds: int = 8
    acquisition_batch_size: int = 20
    class_balance_power: float = 1.0
    n_seed_labeled: int = 80

    lr_head: float = 1e-3
    lr_lora: float = 1e-4
    weight_decay: float = 3e-3
    epochs_per_round: int = 15
    batch_size: int = 32
    focal_gamma: float = 2.0

    early_stop_patience: int = 3
    early_stop_min_delta: float = 1e-4

    seed: int = 42   # overwritten per-run in the seed loop below

cfg = Config()
SEEDS = [42, 123, 2024]   # 3 seeds -- minimum for a mean+/-std claim
print(cfg)
print('Seeds to run:', SEEDS)

Config(image_size=224, num_classes=8, rare_class_idx=2, backbone_name='facebook/dinov2-small', lora_r=8, lora_alpha=16, lora_dropout=0.05, lora_target_modules=('query', 'value'), hidden_dim=256, mc_dropout_p=0.3, mc_passes=20, n_al_rounds=8, acquisition_batch_size=20, class_balance_power=1.0, n_seed_labeled=80, lr_head=0.001, lr_lora=0.0001, weight_decay=0.003, epochs_per_round=15, batch_size=32, focal_gamma=2.0, early_stop_patience=3, early_stop_min_delta=0.0001, seed=42)
Seeds to run: [42, 123, 2024]


## 3. Backbone, Bayesian head, loss, data helpers, acquisition functions (unchanged from Phase 7)

In [ ]:
processor = AutoImageProcessor.from_pretrained(cfg.backbone_name)

def build_backbone(cfg: Config):
    backbone = AutoModel.from_pretrained(cfg.backbone_name)
    for p in backbone.parameters():
        p.requires_grad = False
    lora_config = LoraConfig(r=cfg.lora_r, lora_alpha=cfg.lora_alpha, lora_dropout=cfg.lora_dropout,
                              target_modules=list(cfg.lora_target_modules), bias='none')
    return get_peft_model(backbone, lora_config)


class AlwaysOnDropout(nn.Dropout):
    def forward(self, x):
        return F.dropout(x, p=self.p, training=True)


class BayesianHead(nn.Module):
    def __init__(self, embed_dim, hidden_dim, num_classes, p):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim), nn.GELU(), AlwaysOnDropout(p),
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(), AlwaysOnDropout(p),
            nn.Linear(hidden_dim, num_classes),
        )
    def forward(self, embeddings):
        return self.net(embeddings)


class RareCellModel(nn.Module):
    def __init__(self, backbone, cfg: Config):
        super().__init__()
        self.backbone = backbone
        embed_dim = backbone.config.hidden_size
        self.head = BayesianHead(embed_dim, cfg.hidden_dim, cfg.num_classes, cfg.mc_dropout_p)
    def embed(self, pixel_values):
        return self.backbone(pixel_values=pixel_values).last_hidden_state[:, 0, :]
    def forward(self, pixel_values):
        return self.head(self.embed(pixel_values))


def class_balanced_focal_loss(logits, targets, class_counts, gamma=2.0, beta=0.999, eps=1e-8):
    num_classes = logits.shape[1]
    effective_num = 1.0 - np.power(beta, class_counts)
    weights = (1.0 - beta) / np.clip(effective_num, eps, None)
    weights = weights / weights.sum() * num_classes
    weights = torch.tensor(weights, dtype=torch.float32, device=logits.device)
    log_probs = F.log_softmax(logits, dim=1)
    probs = log_probs.exp()
    ce = F.nll_loss(log_probs, targets, weight=weights, reduction='none')
    pt = probs.gather(1, targets.unsqueeze(1)).squeeze(1)
    focal_term = (1 - pt).clamp(min=eps) ** gamma
    return (focal_term * ce).mean()


class HFProcessorWrapper:
    def __init__(self, processor): self.processor = processor
    def __call__(self, pil_img):
        return self.processor(images=pil_img, return_tensors='pt')['pixel_values'][0]

def build_real_datasets(data_dir, processor):
    tfm = HFProcessorWrapper(processor)
    train_pool = ImageFolder(os.path.join(data_dir, 'train'), transform=tfm)
    val_set = ImageFolder(os.path.join(data_dir, 'val'), transform=tfm)
    test_set = ImageFolder(os.path.join(data_dir, 'test'), transform=tfm)
    return train_pool, val_set, test_set

def make_loader(dataset, indices, batch_size, shuffle):
    return DataLoader(Subset(dataset, indices), batch_size=batch_size, shuffle=shuffle,
                       num_workers=2, drop_last=False)

def get_labels(dataset, indices):
    return np.array(dataset.targets)[indices]


@torch.no_grad()
def mc_dropout_predict(model, loader, T, device):
    model.eval()
    all_probs = []
    for t in range(T):
        batch_probs = []
        for pixel_values, _ in loader:
            pixel_values = pixel_values.to(device)
            probs = F.softmax(model(pixel_values), dim=1)
            batch_probs.append(probs.cpu().numpy())
        all_probs.append(np.concatenate(batch_probs, axis=0))
    return np.stack(all_probs, axis=0)


def bald_scores(mc_probs):
    eps = 1e-12
    mean_probs = mc_probs.mean(axis=0)
    predictive_entropy = -(mean_probs * np.log(mean_probs + eps)).sum(axis=1)
    per_pass_entropy = -(mc_probs * np.log(mc_probs + eps)).sum(axis=2)
    return predictive_entropy - per_pass_entropy.mean(axis=0)

def entropy_scores(mc_probs):
    mean_probs = mc_probs.mean(axis=0)
    eps = 1e-12
    return -(mean_probs * np.log(mean_probs + eps)).sum(axis=1)

def margin_scores(mc_probs):
    mean_probs = mc_probs.mean(axis=0)
    sorted_probs = np.sort(mean_probs, axis=1)
    return -(sorted_probs[:, -1] - sorted_probs[:, -2])

def class_balanced_bald(mc_probs, labeled_class_counts, power=1.0):
    scores = bald_scores(mc_probs)
    mean_probs = mc_probs.mean(axis=0)
    pred_class = mean_probs.argmax(axis=1)
    counts = labeled_class_counts.astype(np.float64) + 1.0
    inv_freq = (1.0 / counts) ** power
    inv_freq = inv_freq / inv_freq.sum() * len(counts)
    return scores * inv_freq[pred_class]

def class_balanced_bald_v2(mc_probs, labeled_class_counts, power=1.0):
    scores = bald_scores(mc_probs)
    mean_probs = mc_probs.mean(axis=0)
    counts = labeled_class_counts.astype(np.float64) + 1.0
    inv_freq = (1.0 / counts) ** power
    inv_freq = inv_freq / inv_freq.sum() * len(counts)
    weights = (mean_probs * inv_freq[None, :]).sum(axis=1)
    return scores * weights

## 4. NEW — confidence diagnostic
Prints the mean and distribution of the model's max-softmax-probability (its "confidence") over
the unlabeled pool at round 0, right before the acquisition-strategy comparison runs. High mean
confidence (>0.85-0.9) here would confirm that `class_balanced_bald_v2` and `class_balanced_bald`
are mathematically forced to (near-)coincide, explaining Phase 7's identical 30.0% bars.

In [ ]:
def confidence_diagnostic(mc_probs, label=''):
    mean_probs = mc_probs.mean(axis=0)
    max_conf = mean_probs.max(axis=1)
    print(f'\n[Confidence diagnostic {label}]')
    print(f'  mean max-prob : {max_conf.mean():.4f}')
    print(f'  median        : {np.median(max_conf):.4f}')
    print(f'  % above 0.85  : {100.0*(max_conf > 0.85).mean():.1f}%')
    print(f'  % above 0.95  : {100.0*(max_conf > 0.95).mean():.1f}%')
    if max_conf.mean() > 0.85:
        print('  -> HIGH confidence: class_balanced_bald and class_balanced_bald_v2 are expected'
              ' to nearly coincide (soft-weighted score collapses toward the argmax-only score).')
    else:
        print('  -> Confidence is not saturated: any observed difference between the OLD and NEW'
              ' acquisition functions in this run is not explained by this mechanism alone.')
    return {'mean_max_prob': float(max_conf.mean()), 'median_max_prob': float(np.median(max_conf)),
            'pct_above_0.85': float(100.0*(max_conf > 0.85).mean()),
            'pct_above_0.95': float(100.0*(max_conf > 0.95).mean())}

## 5. Training loop with early stopping (unchanged from Phase 4 v2 / Phase 7)

In [ ]:
@torch.no_grad()
def _quick_val_loss(model, val_set, val_indices, cfg, T=3):
    model.eval()
    loader = make_loader(val_set, val_indices, cfg.batch_size, shuffle=False)
    total_loss, n = 0.0, 0
    for pixel_values, targets in loader:
        pixel_values, targets = pixel_values.to(device), targets.to(device)
        probs_accum = 0.0
        for _ in range(T):
            probs_accum = probs_accum + F.softmax(model(pixel_values), dim=1)
        mean_probs = probs_accum / T
        loss = F.nll_loss(torch.log(mean_probs.clamp_min(1e-8)), targets, reduction='mean')
        total_loss += loss.item() * len(targets); n += len(targets)
    return total_loss / max(n, 1)


def train_round(model, train_dataset, labeled_idx, cfg, round_id, val_set, val_indices):
    labels = get_labels(train_dataset, labeled_idx)
    class_counts = np.bincount(labels, minlength=cfg.num_classes).astype(np.float64)
    loader = make_loader(train_dataset, labeled_idx, cfg.batch_size, shuffle=True)

    lora_params = [p for n, p in model.named_parameters() if p.requires_grad and 'lora' in n]
    head_params = [p for n, p in model.named_parameters() if p.requires_grad and 'lora' not in n]
    optimizer = torch.optim.AdamW([
        {'params': head_params, 'lr': cfg.lr_head},
        {'params': lora_params, 'lr': cfg.lr_lora},
    ], weight_decay=cfg.weight_decay)

    best_val_loss, best_state, patience_ctr = float('inf'), None, 0
    history = []
    for epoch in range(cfg.epochs_per_round):
        model.train()
        epoch_loss, n_batches = 0.0, 0
        for pixel_values, targets in loader:
            pixel_values, targets = pixel_values.to(device), targets.to(device)
            optimizer.zero_grad()
            loss = class_balanced_focal_loss(model(pixel_values), targets, class_counts, gamma=cfg.focal_gamma)
            loss.backward(); optimizer.step()
            epoch_loss += loss.item(); n_batches += 1
        history.append(epoch_loss / max(n_batches, 1))

        val_loss = _quick_val_loss(model, val_set, val_indices, cfg)
        if val_loss < best_val_loss - cfg.early_stop_min_delta:
            best_val_loss, patience_ctr = val_loss, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            patience_ctr += 1
        if patience_ctr >= cfg.early_stop_patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    print(f'[Round {round_id}] labeled={len(labeled_idx)} final_train_loss={history[-1]:.4f} best_val_loss={best_val_loss:.4f}')
    return history

## 6. Evaluation metrics + `run_experiment()` (adds confidence diagnostic + accepts a `seed` arg)

In [ ]:
def expected_calibration_error(probs, labels, n_bins=15):
    confidences = probs.max(axis=1)
    predictions = probs.argmax(axis=1)
    accuracies = (predictions == labels).astype(np.float64)
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        mask = (confidences > lo) & (confidences <= hi)
        if mask.sum() == 0: continue
        ece += (mask.sum() / len(labels)) * abs(accuracies[mask].mean() - confidences[mask].mean())
    return float(ece)


@torch.no_grad()
def evaluate(model, dataset, indices, cfg, T_eval=10):
    loader = make_loader(dataset, indices, cfg.batch_size, shuffle=False)
    labels = get_labels(dataset, indices)
    mc_probs = mc_dropout_predict(model, loader, T=T_eval, device=device)
    mean_probs = mc_probs.mean(axis=0)
    preds = mean_probs.argmax(axis=1)
    macro_f1 = f1_score(labels, preds, average='macro', zero_division=0)
    rare_present = cfg.rare_class_idx in np.unique(labels)
    rare_f1 = f1_score(labels, preds, labels=[cfg.rare_class_idx], average='macro', zero_division=0) if rare_present else float('nan')
    balanced_acc = balanced_accuracy_score(labels, preds)
    rare_binary_true = (labels == cfg.rare_class_idx).astype(int)
    rare_auprc = average_precision_score(rare_binary_true, mean_probs[:, cfg.rare_class_idx]) if rare_binary_true.sum() > 0 else float('nan')
    ece = expected_calibration_error(mean_probs, labels)
    return {'macro_f1': float(macro_f1), 'rare_class_f1': float(rare_f1),
            'balanced_accuracy': float(balanced_acc), 'rare_auprc': float(rare_auprc), 'ece': float(ece)}


def compare_acquisition_strategies(model, train_pool, unlabeled_idx, labeled_idx, cfg, label=''):
    pool_loader = make_loader(train_pool, unlabeled_idx, cfg.batch_size, shuffle=False)
    mc_probs = mc_dropout_predict(model, pool_loader, T=cfg.mc_passes, device=device)
    conf_diag = confidence_diagnostic(mc_probs, label=label)

    labeled_class_counts = np.bincount(get_labels(train_pool, labeled_idx), minlength=cfg.num_classes)
    strategies = {
        'random': np.random.RandomState(0).rand(len(unlabeled_idx)),
        'entropy': entropy_scores(mc_probs),
        'margin': margin_scores(mc_probs),
        'bald': bald_scores(mc_probs),
        'class_balanced_bald_OLD': class_balanced_bald(mc_probs, labeled_class_counts, cfg.class_balance_power),
        'class_balanced_bald_NEW': class_balanced_bald_v2(mc_probs, labeled_class_counts, cfg.class_balance_power),
    }
    pool_labels = get_labels(train_pool, unlabeled_idx)
    n_rare = int((pool_labels == cfg.rare_class_idx).sum())
    print(f'\nAcquisition comparison {label} | unlabeled={len(unlabeled_idx)} | rare available={n_rare}')
    print(f"{'strategy':<26}{'rare% in top-B':>18}")
    B = min(cfg.acquisition_batch_size, len(unlabeled_idx))
    result = {}
    for name, scores in strategies.items():
        top_b = np.argsort(-scores)[:B]
        rare_pct = 100.0 * (pool_labels[top_b] == cfg.rare_class_idx).mean()
        result[name] = rare_pct
        print(f'{name:<26}{rare_pct:>17.1f}%')
    return result, conf_diag


def run_experiment(data_dir, out_subdir, cfg, seed):
    cfg.seed = seed
    set_seed(seed)
    out_dir = os.path.join(OUT_DIR, out_subdir, f'seed_{seed}')
    os.makedirs(out_dir, exist_ok=True)
    print(f'\n{"="*90}\nRUNNING: {out_subdir} | seed={seed}  (data: {data_dir})\n{"="*90}')

    backbone = build_backbone(cfg).to(device)
    model = RareCellModel(backbone, cfg).to(device)

    train_pool, val_set, test_set = build_real_datasets(data_dir, processor)
    all_indices = np.arange(len(train_pool))
    train_labels_all = get_labels(train_pool, all_indices)

    rng = np.random.RandomState(seed)
    rare_indices = all_indices[train_labels_all == cfg.rare_class_idx]
    common_indices = all_indices[train_labels_all != cfg.rare_class_idx]
    seed_rare = rng.choice(rare_indices, size=min(3, len(rare_indices)), replace=False)
    seed_common = rng.choice(common_indices, size=cfg.n_seed_labeled - len(seed_rare), replace=False)
    labeled_idx = list(np.concatenate([seed_rare, seed_common]))
    unlabeled_idx = list(np.setdiff1d(all_indices, labeled_idx))

    val_indices, test_indices = np.arange(len(val_set)), np.arange(len(test_set))
    results_log, acquisition_comparison, confidence_diag = [], None, None

    for round_id in range(cfg.n_al_rounds):
        t0 = time.time()
        train_round(model, train_pool, labeled_idx, cfg, round_id, val_set, val_indices)
        val_metrics = evaluate(model, val_set, val_indices, cfg)
        test_metrics = evaluate(model, test_set, test_indices, cfg)
        results_log.append({'round': round_id, 'n_labeled': len(labeled_idx), 'val': val_metrics, 'test': test_metrics})

        if len(unlabeled_idx) == 0 or round_id == cfg.n_al_rounds - 1:
            print(f'[Round {round_id}] done in {time.time()-t0:.1f}s (final)')
            break

        if round_id == 0:
            acquisition_comparison, confidence_diag = compare_acquisition_strategies(
                model, train_pool, unlabeled_idx, labeled_idx, cfg, label=f'(seed {seed}, round 0)')

        pool_loader = make_loader(train_pool, unlabeled_idx, cfg.batch_size, shuffle=False)
        mc_probs = mc_dropout_predict(model, pool_loader, T=cfg.mc_passes, device=device)
        labeled_class_counts = np.bincount(get_labels(train_pool, labeled_idx), minlength=cfg.num_classes)
        scores = class_balanced_bald_v2(mc_probs, labeled_class_counts, power=cfg.class_balance_power)
        B = min(cfg.acquisition_batch_size, len(unlabeled_idx))
        acquired = [unlabeled_idx[i] for i in np.argsort(-scores)[:B]]
        labeled_idx.extend(acquired)
        unlabeled_idx = list(np.setdiff1d(unlabeled_idx, acquired))
        print(f'[Round {round_id}] done in {time.time()-t0:.1f}s')

    with open(os.path.join(out_dir, 'al_results_log.json'), 'w') as f:
        json.dump(results_log, f, indent=2)
    with open(os.path.join(out_dir, 'acquisition_comparison_round0.json'), 'w') as f:
        json.dump(acquisition_comparison, f, indent=2)
    with open(os.path.join(out_dir, 'confidence_diagnostic.json'), 'w') as f:
        json.dump(confidence_diag, f, indent=2)

    return results_log, acquisition_comparison, confidence_diag

## 7. Run 3 seeds on same-batch data (`data_v2/`) — primary claim

In [ ]:
all_results_samebatch = {}
all_acq_samebatch = {}
all_conf_samebatch = {}

for seed in SEEDS:
    results_log, acq_cmp, conf_diag = run_experiment(DATA_DIR_SAMEBATCH, 'phase9_samebatch', cfg, seed)
    all_results_samebatch[seed] = results_log
    all_acq_samebatch[seed] = acq_cmp
    all_conf_samebatch[seed] = conf_diag


RUNNING: phase9_samebatch | seed=42  (data: /content/drive/MyDrive/cse754/data_v2)


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

[Round 0] labeled=80 final_train_loss=0.0557 best_val_loss=1.2850

[Confidence diagnostic (seed 42, round 0)]
  mean max-prob : 0.4800
  median        : 0.4519
  % above 0.85  : 2.0%
  % above 0.95  : 0.0%
  -> Confidence is not saturated: any observed difference between the OLD and NEW acquisition functions in this run is not explained by this mechanism alone.

Acquisition comparison (seed 42, round 0) | unlabeled=2735 | rare available=12
strategy                      rare% in top-B
random                                  0.0%
entropy                                 0.0%
margin                                  0.0%
bald                                    0.0%
class_balanced_bald_OLD                30.0%
class_balanced_bald_NEW                30.0%
[Round 0] done in 982.3s
[Round 1] labeled=100 final_train_loss=0.1553 best_val_loss=1.1651
[Round 1] done in 519.2s
[Round 2] labeled=120 final_train_loss=0.0949 best_val_loss=1.2170
[Round 2] done in 511.1s
[Round 3] labeled=140 final_trai

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

[Round 0] labeled=80 final_train_loss=0.0958 best_val_loss=1.3469

[Confidence diagnostic (seed 123, round 0)]
  mean max-prob : 0.5003
  median        : 0.4801
  % above 0.85  : 1.4%
  % above 0.95  : 0.0%
  -> Confidence is not saturated: any observed difference between the OLD and NEW acquisition functions in this run is not explained by this mechanism alone.

Acquisition comparison (seed 123, round 0) | unlabeled=2735 | rare available=12
strategy                      rare% in top-B
random                                  0.0%
entropy                                 0.0%
margin                                  0.0%
bald                                    0.0%
class_balanced_bald_OLD                 0.0%
class_balanced_bald_NEW                 5.0%
[Round 0] done in 959.3s
[Round 1] labeled=100 final_train_loss=0.0227 best_val_loss=1.4507
[Round 1] done in 549.3s
[Round 2] labeled=120 final_train_loss=0.0069 best_val_loss=1.3695
[Round 2] done in 573.0s
[Round 3] labeled=140 final_tr

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

[Round 0] labeled=80 final_train_loss=0.0697 best_val_loss=1.2195

[Confidence diagnostic (seed 2024, round 0)]
  mean max-prob : 0.5163
  median        : 0.4879
  % above 0.85  : 5.3%
  % above 0.95  : 0.6%
  -> Confidence is not saturated: any observed difference between the OLD and NEW acquisition functions in this run is not explained by this mechanism alone.

Acquisition comparison (seed 2024, round 0) | unlabeled=2735 | rare available=12
strategy                      rare% in top-B
random                                  0.0%
entropy                                 0.0%
margin                                  0.0%
bald                                    0.0%
class_balanced_bald_OLD                10.0%
class_balanced_bald_NEW                20.0%
[Round 0] done in 961.1s
[Round 1] labeled=100 final_train_loss=0.0125 best_val_loss=1.2384


## 8. (OPTIONAL, not run by default) Cross-batch multi-seed
Uncomment and run this cell only if you have compute budget left -- it roughly doubles total
runtime. The same-batch results above are sufficient to validate the acquisition-function claim;
this cell would additionally validate the generalization-gap claim across seeds instead of the
single seed Phase 7 used.

In [ ]:
RUN_CROSSBATCH_MULTISEED = False   # flip to True if you have compute budget left

if RUN_CROSSBATCH_MULTISEED:
    all_results_crossbatch = {}
    all_acq_crossbatch = {}
    all_conf_crossbatch = {}
    for seed in SEEDS:
        results_log, acq_cmp, conf_diag = run_experiment(DATA_DIR_CROSSBATCH, 'phase9_crossbatch', cfg, seed)
        all_results_crossbatch[seed] = results_log
        all_acq_crossbatch[seed] = acq_cmp
        all_conf_crossbatch[seed] = conf_diag
else:
    print('Skipped (RUN_CROSSBATCH_MULTISEED=False). Set to True to run.')

## 9. Aggregate across seeds: mean +/- std for headline metrics + acquisition comparison

In [ ]:
def final_test_metrics(results_log):
    return results_log[-1]['test']

metrics_by_seed = {seed: final_test_metrics(log) for seed, log in all_results_samebatch.items()}
metric_names = ['macro_f1', 'rare_class_f1', 'rare_auprc', 'ece']

print(f"{'metric':<18}{'mean':>10}{'std':>10}{'  (values across seeds)'}")
agg_summary = {}
for m in metric_names:
    vals = [metrics_by_seed[s][m] for s in SEEDS]
    mean_v, std_v = float(np.mean(vals)), float(np.std(vals))
    agg_summary[m] = {'mean': mean_v, 'std': std_v, 'values': vals}
    print(f'{m:<18}{mean_v:>10.4f}{std_v:>10.4f}   {[round(v,4) for v in vals]}')

# acquisition strategy comparison, aggregated
strategy_names = list(next(iter(all_acq_samebatch.values())).keys())
print(f"\n{'strategy':<26}{'mean %':>10}{'std %':>10}")
acq_agg_summary = {}
for name in strategy_names:
    vals = [all_acq_samebatch[s][name] for s in SEEDS]
    mean_v, std_v = float(np.mean(vals)), float(np.std(vals))
    acq_agg_summary[name] = {'mean': mean_v, 'std': std_v, 'values': vals}
    print(f'{name:<26}{mean_v:>10.1f}{std_v:>10.1f}')

# confidence diagnostic, aggregated
conf_vals = [all_conf_samebatch[s]['mean_max_prob'] for s in SEEDS]
print(f'\nMean max-prob at round 0 across seeds: {np.mean(conf_vals):.4f} +/- {np.std(conf_vals):.4f}')

with open(os.path.join(OUT_DIR, 'phase9_multiseed_summary.json'), 'w') as f:
    json.dump({'metrics': agg_summary, 'acquisition': acq_agg_summary,
               'confidence_mean_across_seeds': float(np.mean(conf_vals)),
               'seeds': SEEDS}, f, indent=2)
print('\nSaved to', os.path.join(OUT_DIR, 'phase9_multiseed_summary.json'))

## 10. Plot: mean +/- std annotation efficiency + acquisition comparison with error bars

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

n_labeled_ref = [r['n_labeled'] for r in next(iter(all_results_samebatch.values()))]
for metric_key, marker, label in [('macro_f1', 'o', 'Macro-F1'), ('rare_class_f1', 's', 'Rare-F1')]:
    per_round_vals = []
    for round_idx in range(len(n_labeled_ref)):
        round_vals = [all_results_samebatch[s][round_idx]['test'][metric_key] for s in SEEDS
                      if round_idx < len(all_results_samebatch[s])]
        per_round_vals.append(round_vals)
    means = [np.mean(v) for v in per_round_vals]
    stds = [np.std(v) for v in per_round_vals]
    axes[0].errorbar(n_labeled_ref[:len(means)], means, yerr=stds, marker=marker, capsize=3, label=label)

axes[0].set_xlabel('# Labeled images'); axes[0].set_ylabel('F1')
axes[0].set_title(f'Annotation efficiency, mean +/- std over {len(SEEDS)} seeds')
axes[0].legend(); axes[0].grid(alpha=0.3)

names = strategy_names
means = [acq_agg_summary[n]['mean'] for n in names]
stds = [acq_agg_summary[n]['std'] for n in names]
colors = ['tab:red' if 'OLD' in n else 'tab:green' if 'NEW' in n else 'tab:gray' for n in names]
axes[1].bar(names, means, yerr=stds, capsize=4, color=colors)
axes[1].set_ylabel('% rare-class in top-B (round 0)')
axes[1].set_title(f'Acquisition comparison, mean +/- std over {len(SEEDS)} seeds')
axes[1].tick_params(axis='x', rotation=30)
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plot_path = os.path.join(OUT_DIR, 'phase9_multiseed_plots.png')
plt.savefig(plot_path, dpi=150)
plt.show()
print('Saved to', plot_path)

## Notes / next steps
- If the `class_balanced_bald_NEW` bar's error bar in Section 10 overlaps heavily with
  `class_balanced_bald_OLD`'s, the two functions are statistically indistinguishable at this
  sample size -- report that plainly, alongside the confidence-diagnostic explanation from
  Section 4/6, rather than claiming a fix that isn't distinguishable from noise.
- Feed `phase9_multiseed_summary.json` into **Phase 11** (updated consolidated report) alongside
  Phase 10's linear-probe results.
- If `RUN_CROSSBATCH_MULTISEED` was left `False`, Phase 7's single-seed generalization-gap number
  is still the only one on record -- flag this explicitly in your report's limitations.